# PointCNN inference: point cloud -> segmentation labels

Takes a preprocessed `.ply` (output of `Point_Cloud_Preprocessing.ipynb`) and produces a `predicted_3.txt`-style file (`x, y, z, label`) for `Weld_Recognition&Grinding_Path_Planning(PointCNN).ipynb`.

Each pass is a farthest-point sample (`torch_geometric.nn.fps`) of `n_points` (2048), not `np.random.choice` like `old/Weld_Recognition&Grinding_Path_Planning(PointNet).ipynb` (cell 6) used -- FPS is what `PointCNN_Training_Functions/dataset/wireharness_dataset.py` (the actual PointCNN training code) uses to build a training sample, so matching it keeps inference in-distribution with training. Unlike PointNet, there's no mean-centering / unit-sphere normalization either: `wireharness_dataset.py` loads raw millimeter coordinates directly with no normalization step, so the checkpoint expects raw coordinates too.

**Voting across multiple passes** (not the original PointNet notebook's single concatenated sweep): tested on `test.ply` and found two concrete problems with a single sweep -- (1) independent random FPS draws only covered 56% of the scan at all (the rest never got a prediction, purely from sampling overlap/gaps), and (2) of the points seen more than once, a real fraction flip between "bead" and "not bead" depending only on which other points happened to land in that particular sample (XConv's decision is context-dependent). Voting `passes` FPS samples per point and thresholding the bead fraction fixes both: more passes -> better coverage, and disagreement across independent contexts gets resolved by majority instead of left as noise.

Also route the input through `Point_Cloud_Preprocessing.ipynb` first rather than feeding `test.ply` directly: the raw scan has a real floating noise cluster (depth-camera artifact, physically disconnected from the actual surface) that gets misclassified as bead because it's a compact raised blob shaped enough like one to fool the model. `Point_Cloud_Preprocessing.ipynb`'s statistical outlier removal (previously disabled in the original notebook, re-enabled here) filters it out before it ever reaches PointCNN.

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import os
os.environ.pop('WAYLAND_DISPLAY', None)  # Open3D's GLFW window fails to open under native Wayland on this machine (GLEW init error) -- force XWayland instead
os.environ['XDG_SESSION_TYPE'] = 'x11'
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
from torch_geometric.nn import fps

sys.path.insert(0, os.path.join(os.getcwd(), "PointCNN_Training_Functions"))

# Load PointCNN model

In [ ]:
from model.pointcnn_seg import PointCNNSeg

ckpt_path = "PointCNN_weight(Torch)/pointcnn-best_8_2048.ckpt"  # best val accuracy (98.72%), see pointcnn_result.txt
num_classes = 2
n_points = 2048  # matches the checkpoint name (batch_8_2048) and its training sample size

device = "cuda" if torch.cuda.is_available() else "cpu"
model = PointCNNSeg.load_from_checkpoint(ckpt_path, map_location=device, num_classes=num_classes, weight_balance=None)
model.eval()
model.to(device)
print("Model loaded on", device)

# Load point cloud

In [ ]:
time1 = time.time()

point_file = 'outputs/preprocessed.ply'  # output of Point_Cloud_Preprocessing.ipynb
# point_file = 'test.ply'  # raw camera output -- works, but includes scanner noise Point_Cloud_Preprocessing.ipynb would filter out
# point_file = 'outputs/predicted_3.txt'  # if pointing at an already-labeled file (e.g. a validation sample), the
                                   # label column is used to print accuracy below, same as the old PointNet notebook

if point_file.endswith('.ply'):
    pcd = o3d.io.read_point_cloud(point_file)
    point_cloud = np.asarray(pcd.points, dtype=np.float32)
    truth_label = None
else:
    cloud = np.loadtxt(point_file)
    point_cloud = np.array(cloud[:, :3], dtype=np.float32)
    truth_label = np.array(cloud[:, 3], dtype=np.int64) if cloud.shape[1] > 3 else None

num_points = point_cloud.shape[0]
print(f"Loaded {num_points} points from {point_file}")

# PointCNN segmentation

In [ ]:
# Vote over multiple FPS passes instead of a single batch sweep: one pass only "sees" ~2%
# of a real scan (fixed 2048-point sample size from training), and the same point can get a
# different label in different passes since XConv's decision depends on which other points
# happen to be in that pass's sample (verified: 105 points flipped label between passes in
# an earlier single-sweep run). Voting also naturally increases coverage -- n_batches
# independent random draws only covered 56% of test.ply due to overlap/gaps.
passes = 60
vote_threshold = 0.5
batch_size = 16

coords_t = torch.tensor(point_cloud).to(device)  # fps() is much faster on GPU (~3s vs ~11s per call on 500k points)
n_total = coords_t.shape[0]

bead_votes = np.zeros(n_total, dtype=np.int32)
seen_votes = np.zeros(n_total, dtype=np.int32)
truth_by_index = truth_label if truth_label is not None else None

done = 0
while done < passes:
    b = min(batch_size, passes - done)
    idx_list, sample_list = [], []
    for _ in range(b):
        if n_total > n_points:
            idx = fps(coords_t, ratio=n_points / n_total)[:n_points]
            sample_list.append(coords_t[idx])
        else:
            pad = n_points - n_total
            idx = torch.arange(n_total)
            sample_list.append(torch.cat([coords_t, coords_t[:pad]], dim=0))
        idx_list.append(idx)

    batch = torch.stack(sample_list, dim=0).to(device)
    with torch.no_grad():
        logits = model(batch)  # (b, num_classes, n_points)
        pred = torch.argmax(logits, dim=1).cpu().numpy()  # (b, n_points)

    for i, idx in enumerate(idx_list):
        idx_np = idx.cpu().numpy()[: pred.shape[1]]
        seen_votes[idx_np] += 1
        bead_votes[idx_np] += (pred[i] == 1).astype(np.int32)

    done += b

seen_mask = seen_votes > 0
all_labels_full = np.zeros(n_total, dtype=np.int64)
with np.errstate(invalid="ignore"):
    frac = np.where(seen_mask, bead_votes / np.maximum(seen_votes, 1), 0.0)
all_labels_full[seen_mask & (frac >= vote_threshold)] = 1

all_point_clouds = point_cloud[seen_mask]
all_labels = all_labels_full[seen_mask]

print("Point clouds shape:", all_point_clouds.shape)
print("Labels shape:", all_labels.shape)
print(f"Coverage: {seen_mask.sum()} / {n_total} points seen at least once ({100*seen_mask.sum()/n_total:.1f}%)")

if truth_by_index is not None:
    all_truth_labels = truth_by_index[seen_mask]
    correct_predictions = np.sum(all_labels == all_truth_labels)
    accuracy = correct_predictions / all_labels.shape[0]
    print(f"Correctly classified points: {correct_predictions}")
    print(f"Total points: {all_labels.shape[0]}")
    print(f"Classification accuracy: {accuracy:.4f}")

time2 = time.time()
t1 = np.round((time2 - time1), 4)
print("Weld bead recognition time: ", t1, "s")

# View segmentation results

In [ ]:
points_label0 = all_point_clouds[all_labels == 0]
points_label1 = all_point_clouds[all_labels == 1]
print(f"label 0 (not bead): {len(points_label0)}, label 1 (bead): {len(points_label1)}")

fig = plt.figure(figsize=(16, 7))

# 3D view
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(points_label0[:, 0], points_label0[:, 1], points_label0[:, 2], c='lightblue', s=1, alpha=0.15, label='label 0 (work piece)')
ax1.scatter(points_label1[:, 0], points_label1[:, 1], points_label1[:, 2], c='red', s=10, label='label 1 (bead)')
ax1.set_xlabel('X(mm)'); ax1.set_ylabel('Y(mm)'); ax1.set_zlabel('Z(mm)')
ax1.set_title('3D view')
ax1.legend()

# Top-down (X-Y) view -- easiest way to check the bead traces the expected shape
ax2 = fig.add_subplot(122)
ax2.scatter(points_label0[:, 0], points_label0[:, 1], c='lightblue', s=1, alpha=0.15, label='label 0 (work piece)')
ax2.scatter(points_label1[:, 0], points_label1[:, 1], c='red', s=10, label='label 1 (bead)')
ax2.set_xlabel('X(mm)'); ax2.set_ylabel('Y(mm)')
ax2.set_title('Top-down (X-Y) view')
ax2.axis('equal')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Same Open3D interactive view as the original notebooks (requires a real display)
pcd_label0 = o3d.geometry.PointCloud()
pcd_label0.points = o3d.utility.Vector3dVector(points_label0)
pcd_label0.paint_uniform_color([0, 0, 1])  # blue

pcd_label1 = o3d.geometry.PointCloud()
pcd_label1.points = o3d.utility.Vector3dVector(points_label1)
pcd_label1.paint_uniform_color([1, 0, 0])  # red

combined_pcd = pcd_label0 + pcd_label1

vis = o3d.visualization.Visualizer()
vis.create_window()
vis.add_geometry(combined_pcd)
axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=10)
view_ctl = vis.get_view_control()
view_ctl.set_up([0, 0, 1])
view_ctl.set_lookat([0, 0, 0])
view_ctl.set_front([1, 1, 1])
vis.run()
vis.destroy_window()

# Save predicted labels

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)
out_path = 'outputs/predicted_3.txt'
out = np.column_stack([all_point_clouds, all_labels])
np.savetxt(out_path, out, fmt="%.6f")
print(f"Wrote {out_path}: {len(out)} points ({len(points_label1)} labeled bead)")